# 基于机器学习与 PROMETHEE 综合评价的股票多指标建模与选股分析

**课程：** 大数据决策 / 大数据建模相关课程  
**数据对象：** 多期股票财务因子与未来收益标签  
**方法关键词：** 数据审计、标签前移验证、防泄漏处理、机器学习排序、RankIC、PROMETHEE、熵权法、Hybrid Score、Top-K 回测  
**最终推荐方案：** `hybrid_score_50_50 + Top20`

本 Notebook 是报告初稿型文档，主要整理已有 `outputs` 目录中的审计、建模、评分和回测结果。本文不在此处重新训练模型、不重新清洗数据，也不重新构造标签。


# 摘要

本文围绕多期股票财务因子与下一期收益标签，构建了一个从数据审计、标签验证、防泄漏处理、特征治理、机器学习排序、PROMETHEE 综合评价、Hybrid Score 融合到 Top-K 回测的完整建模流程。研究目标并不是单纯预测股票未来收益率的绝对数值，而是在财务因子弱信号条件下，构建一个可解释、可复现的横截面排序和多指标辅助选股框架。

在数据处理阶段，本文首先验证原始“月平均收益率”字段已经对应下一期收益，并将正式监督学习标签命名为 `future_return_1q`，避免重复前移。随后，收益类字段、标签字段、ID 字段和 meta 字段均被剔除出特征矩阵；时间切分采用 `train=quarter_idx 1–21`、`valid=22–25`、`test=26–30` 的严格时间顺序方案，缺失值填补、缩尾和标准化参数均只在训练窗口拟合。

在模型阶段，本文比较了 Ridge、Lasso、ElasticNet、Random Forest、HistGradientBoosting、XGBoost 和 LightGBM 等方法。结果显示，模型的 R² 整体较弱，说明其不适合被解释为高精度收益率预测模型；但 RankIC、分层收益和 Top-K 回测显示模型存在一定横截面排序能力。进一步地，本文使用熵权法与 PROMETHEE 建立基本面综合评价，并用固定权重将 PROMETHEE 与机器学习排序信号融合为 Hybrid Score。

最终，本文推荐 `hybrid_score_50_50 + Top20` 作为主方案。该方案并非单纯追求最高测试收益，而是在解释性、持仓分散度、课程报告完整性和策略稳健性之间取得折中。`hybrid_score_30_70 + Top10` 可作为收益最优的激进对照方案。评级检验显示高评级组多数情况下优于低评级组，但收益不完全单调，因此评级更适合作为候选池筛选工具，而非严格单调的收益等级体系。


# 1. 研究背景与建模目标

股票收益率预测具有典型的高噪声、非线性、非平稳和弱信号特征。未来收益不仅受到公司基本面影响，还受到市场情绪、政策变化、资金流动、行业轮动、突发事件和宏观环境等多种因素共同驱动。因此，单纯使用财务因子精确预测下一期收益率数值本身难度较高。

基于这一现实约束，本文将建模目标定义为：构建完整的数据处理、模型评分、综合评价和回测框架，以检验多因子体系是否能提供横截面排序信号，并进一步转化为 Top-K 组合。换言之，本文更关注“哪些股票相对更值得进入候选池”，而不是预测每只股票未来收益率的精确点估计。

这一目标也决定了评价指标体系。除 MAE、RMSE、R² 等回归指标外，本文更重视 RankIC、分层收益、Top-K 组合收益、换手率和相对基准表现。最终策略选择不只依据测试集收益最高，而是综合考虑验证集表现、测试集表现、解释性、分散度和可复现性。


# 2. 数据来源与字段体系

本文使用多期股票财务因子、估值因子、成长因子、现金流质量因子以及未来收益标签构建建模面板。经过 P0 阶段的数据盘点与特征角色识别，最终模型使用 21 个基础财务因子，覆盖盈利能力、利润结构、现金流质量、成长性、运营效率、规模和估值等维度。

宏观因子方面，原始任务文档口径为 12 个宏观因子，但实际数据文件中只能识别 11 个具名且可验证的宏观因子。由于第 12 个因子缺少明确字段名、定义和可追溯来源，无法可靠派生，因此本文将其剔除，并在后续报告中统一表述为“实际使用 11 个可验证宏观因子”。这一处理避免了报告中宣称使用 12 个因子但实际只使用 11 个的口径不一致问题。

![图1 数据处理与建模流程](figures/fig01_pipeline_flow.png)

**图表解读。** 图 1 展示了从原始数据到最终评级与回测的完整流程。P0 阶段主要解决数据质量、标签定义、防泄漏和时间切分问题；P1 阶段完成机器学习建模、排序评价与 Top-K 回测；随后通过 PROMETHEE 和 Hybrid Score 将基本面综合评分与模型信号结合。该流程强调先确认数据口径，再进入模型训练与策略评价，避免在标签和特征尚未闭环时直接建模。

![图2 时间切分样本数](figures/fig02_split_sample_count.png)

**图表解读。** 图 2 展示了训练集、验证集和测试集的样本数量。切分完全按照季度时间顺序进行，没有随机打乱。训练集覆盖 `quarter_idx 1–21`，验证集覆盖 `22–25`，测试集覆盖 `26–30`。这种切分方式更符合真实投资场景，也可以降低未来信息泄漏风险。


# 3. 数据处理与 P0 闭环

## 3.1 标签前移验证

原始数据中的标签列为“月平均收益率”。在 P0-F 标签验证阶段，本文对每个相邻季度 `t` 与 `t+1` 进行证券代码匹配，并比较 `t` 期的“月平均收益率”与 `t+1` 期的“区间涨跌幅(3月)”是否一致。验证结果表明，原始“月平均收益率”字段已经与下一期收益一致，因此本文将正式监督学习标签命名为 `future_return_1q`。

这一结论非常重要：后续建模不再对标签进行二次 shift，否则会导致标签错位。同时，`区间涨跌幅(1月)`、`区间涨跌幅(3月)`、`区间涨跌幅(6月)` 等收益类字段均被视为高风险泄漏字段，不进入特征矩阵。

## 3.2 防泄漏字段处理

为了保证模型只使用预测时点可以获得的信息，本文禁止标签字段、收益字段、ID 字段和 meta 字段进入 X。被禁止的字段包括但不限于：`月平均收益率`、`future_return_1q`、`区间涨跌幅(1月)`、`区间涨跌幅(3月)`、`区间涨跌幅(6月)`、`证券代码`、`证券名称`、`quarter_idx`、`label_source`、`label_validation_status`、`is_tail_quarter` 和 `_preprocess_split` 等。

P0 闭环检查中，最终 `return_like_as_factor_count = 0`。这说明收益类字段没有被误当作普通因子进入模型，有助于防止模型通过未来收益字段“作弊”。

## 3.3 缺失值处理

对数值因子，本文采用训练集窗口内中位数填补。中位数对极端值和偏态分布更稳健，适合财务数据中常见的长尾和异常口径。所有填补参数只在训练集上拟合，再应用到验证集和测试集，不使用验证集或测试集统计量拟合填补参数。

## 3.4 异常值与长尾处理

财务比率、增长率、PE、PCF 等字段常出现长尾、极端值或异常财务口径。例如负 PE、负 PCF、爆炸增长率都可能降低模型稳定性。本文采用 winsorize 缩尾处理，并对严重偏态字段使用 `signed_log1p` 变换。缩尾边界同样只基于 train 窗口确定，避免利用未来数据修正训练期分布。

## 3.5 标准化处理

标准化阶段使用 RobustScaler 作为基线方法。RobustScaler 基于中位数和四分位间距进行缩放，对极端值更稳健。与缺失值填补和缩尾相同，标准化参数只在训练集拟合，再应用到验证集和测试集。

## 3.6 时间切分

本文采用固定时间切分：`train = quarter_idx 1–21`，`valid = quarter_idx 22–25`，`test = quarter_idx 26–30`。不允许随机打乱，也不允许尾季度无下一期收益标签的样本进入监督学习评价。这种设计使验证集用于模型和参数选择，测试集仅用于最终 holdout 检验。


# 4. 特征体系与指标方向

最终进入模型和综合评分体系的是 21 个基础财务因子。这些因子可分为盈利能力、利润结构、现金流质量、成长性、运营效率、规模和估值等类别。盈利、成长和现金流质量通常被视为正向指标；PE、PB、PS、PCF 等估值指标被设为负向或谨慎负向指标，因为在其他条件相近时，估值越高可能意味着安全边际越低。

特殊地，`营业外收支净额/利润总额(TTM)` 不能简单设为正向指标。营业外收支占比过高可能意味着利润质量不稳定，因此本文将其作为风险型或区间型指标处理。规模指标也不直接固定为正向或负向：规模较大可能代表稳健性，但也可能带来规模风格暴露，因此 PROMETHEE 中设置了 `with_scale` 和 `without_scale` 两个版本。



## 4.1 原 Excel 因子的方向判别方法

本文对原 Excel 中进入最终特征名单的 21 个因子进行方向判别时，没有使用未来收益率倒推指标方向，也没有根据测试集表现反向调权，而是依据字段名称、财务含义、指标经济解释以及 PROMETHEE 多指标评价的要求进行事前设定。具体来说，判别过程遵循以下原则：

1. **盈利能力、利润率、利润质量和运营效率类指标一般设为正向。** 例如 ROE、ROA、销售净利率、销售毛利率、经营活动净收益占比、营业利润占比和总资产周转率等，数值越高通常表示公司盈利能力、主营业务质量或资产使用效率越好。
2. **成长性指标一般设为正向，但需要缩尾处理。** 营业收入、营业利润、利润总额和 ROE 的一年增长率越高，通常代表成长性越强；但增长率容易出现极端值，因此在建模和评分前进行 winsorize 与稳健标准化。
3. **估值类指标设为负向或谨慎负向。** PE、PB、PS、PCF 越高，通常表示估值越贵、安全边际越低，因此在 PROMETHEE 中按负向准则处理。需要注意，负 PE 或负 PCF 往往对应亏损或现金流异常，并不代表“更便宜”，因此这类字段还需要在异常值治理中进行缩尾和稳健处理。
4. **规模类指标不直接作为绝对正向或负向。** 总市值和流通市值可能代表流动性、稳定性和机构可投资性，但也可能只是规模风格暴露。因此本文设计两套评分口径：`with_scale` 中将规模作为正向稳健性指标，`without_scale` 中将规模视为控制变量并从 PROMETHEE 主评分中剔除。
5. **风险型或区间型指标不能只看原始符号。** `营业外收支净额/利润总额(TTM)` 的正负方向本身并不稳定。营业外收支占比过高说明利润对非主营项目依赖较强，可能降低利润质量，因此本文构造其绝对偏离口径，将 `abs_营业外收支净额/利润总额(TTM)` 作为负向风险指标。

在归一化和 PROMETHEE 输入中，正向指标按“越大越好”处理；负向指标按“越小越好”处理，等价于反向归一化；风险型指标先取绝对偏离或风险暴露，再按负向指标处理；规模指标根据 `with_scale / without_scale` 两个版本分别纳入或剔除。上述方向均为事前规则，不使用 `future_return_1q`、测试集收益或回测结果进行反向修正。

| 原 Excel 因子 | 判别类别 | 评分方向 | 判别依据与处理说明 |
|---|---|---|---|
| 净资产收益率(TTM) | 盈利能力 | 正向 | ROE 越高通常表示股东权益盈利能力越强，作为基本面正向指标。 |
| 总资产报酬率(TTM) | 盈利能力 | 正向 | ROA 越高说明资产整体获利能力越强，作为正向指标。 |
| 销售净利率(TTM) | 盈利能力 | 正向 | 单位收入转化为净利润的能力越强，公司盈利质量通常越好。 |
| 销售毛利率(TTM) | 盈利能力 | 正向 | 毛利率反映产品或服务的基础盈利空间，越高通常越好。 |
| 经营活动净收益/利润总额(TTM) | 利润质量 | 正向 | 利润更多来自经营活动时，可持续性通常更强。 |
| 营业利润/利润总额(TTM) | 利润结构 | 正向 | 营业利润占比越高，说明利润更多来自主营或经常性业务。 |
| 利润总额/营业收入(TTM) | 利润率 | 正向 | 每单位收入创造的利润总额越高，盈利能力越强。 |
| 经营活动产生的现金流量净额/营业收入(TTM) | 现金流质量 | 正向 | 收入转化为经营现金流的能力越强，利润含金量越高。 |
| 经营活动产生的现金流量净额/营业利润(TTM) | 现金流质量 | 正向 | 经营现金流对营业利润覆盖越充分，利润质量通常越好。 |
| 营业收入(1年，增长率) | 成长性 | 正向 | 收入增长反映业务扩张能力，但极端增长率需缩尾。 |
| 营业利润(1年，增长率) | 成长性 | 正向 | 营业利润增长反映主营盈利扩张，但需处理爆炸增长率。 |
| 利润总额(1年，增长率) | 成长性 | 正向 | 利润总额增长代表整体盈利扩张，作为成长性指标。 |
| 净资产收益率(1年，增长率) | 成长性 | 正向 | ROE 增长表示盈利效率改善，但同样需稳健处理极端值。 |
| 总资产周转率(TTM) | 运营效率 | 正向 | 周转率越高说明资产使用效率越高，作为效率类正向指标。 |
| 市盈率PE(TTM) | 估值 | 负向 | PE 越高通常表示盈利估值越贵，安全边际越低；负 PE 视为异常口径处理。 |
| 市净率PB(LF) | 估值 | 负向 | PB 越高表示净资产估值越贵，在其他条件相近时谨慎负向。 |
| 市销率PS(TTM) | 估值 | 负向 | PS 越高表示收入估值越贵，作为估值负向指标。 |
| 市现率PCF(经营性现金流TTM) | 估值 | 负向 | PCF 越高表示现金流估值越贵；负 PCF 可能代表现金流异常。 |
| 总市值(证监会算法) | 规模 | with_scale 正向；without_scale 剔除 | 规模可代表稳定性和流动性，但也可能形成规模风格暴露，因此提供两种口径。 |
| 流通市值 | 规模 | with_scale 正向；without_scale 剔除 | 流通市值越大通常可交易性更好，但主评分中不强制纳入。 |
| 营业外收支净额/利润总额(TTM) | 风险型 / 区间型 | 绝对值负向 | 营业外收支占比过大说明利润质量可能不稳定，因此以绝对偏离作为风险暴露。 |

上述表格说明了每个原 Excel 因子的判别逻辑。整体上，本文的指标方向设定服务于“基本面质量越好、估值越合理、利润越可持续、风险暴露越低”的综合评价目标；同时通过 `without_scale` 版本降低规模风格对主评分的影响。


![图3 特征类别分布](figures/fig03_feature_category_count.png)

**图表解读。** 图 3 展示了最终 21 个因子在不同类别上的分布。可以看到，因子体系并非只集中在单一维度，而是覆盖盈利、成长、现金流、规模和估值等多个方面。多类别特征有助于提升综合评价的覆盖面，但也会带来指标方向设定和口径一致性的要求。

![图4 熵权Top10指标权重](figures/fig04_entropy_top10_weights.png)

**图表解读。** 图 4 展示了训练窗口内熵权最高的前 10 个指标。熵权反映的是指标在样本横截面上的区分度，权重较高说明该指标在不同股票之间差异较大。需要强调的是，熵权不代表经济因果重要性，也不表示该指标一定能预测未来收益。本文将熵权作为 PROMETHEE 综合评价的客观权重输入，而不是作为因果解释。


# 5. 机器学习模型设计与结果

本文首先使用 Ridge、Lasso 和 ElasticNet 作为稀疏线性基线模型，以便获得较强的可解释性和稳定性对照。随后使用 Random Forest、HistGradientBoosting、XGBoost 和 LightGBM 等非线性模型进行比较。模型选择只使用验证集，测试集仅用于最终 holdout 评价，不参与调参或权重选择。

结果显示，Random Forest 是验证集选择出的主模型，Ridge 和 ElasticNet 作为稳健性对照保留。整体来看，模型的回归误差改善有限，R² 较弱或接近 0，说明模型对未来收益率绝对数值的解释能力有限。因此，本文不将模型定位为精确收益率预测模型，而是进一步通过 RankIC、分层收益和 Top-K 回测检验其横截面排序能力。

![图5 模型RMSE与RankIC对比](figures/fig05_model_rmse_rankic_compare.png)

**图表解读。** 图 5 同时展示验证集 RMSE 和 RankIC。RMSE 衡量数值预测误差，RankIC 衡量预测排序与未来收益排序的一致性。图中可以看到，不同模型在 RMSE 上差异并不显著，但 RankIC 存在弱正信号。这说明模型更可能在横截面排序上有价值，而不是在收益率点预测上有强解释力。

![图5b 模型R²对比](figures/fig05b_model_r2_compare.png)

**图表解读。** 图 5b 专门展示不同模型在验证集和测试集上的 R²。若 R² 为负或接近 0，表示模型对未来收益率绝对数值的解释能力较弱，甚至不优于简单均值基准。该结果进一步支持本文的定位：后续评价重点应转向排序能力、分层收益和 Top-K 组合表现，而不是强调收益率精确预测。

![图6 特征重要性Top10](figures/fig06_feature_importance_top10.png)

**图表解读。** 图 6 展示了主模型或平均口径下的重要特征。特征重要性可以帮助理解模型更关注哪些财务维度，但它并不等同于因果关系。对于课程报告而言，该图可用于说明模型并非黑箱地直接输出组合，而是在财务因子体系上学习横截面差异。


# 6. 为什么数据和模型效果可能较差

本节专门讨论数据质量和模型效果较弱的可能原因。首先，金融收益率本身信噪比较低。股票未来收益受到市场情绪、政策变化、流动性、行业轮动和突发事件影响，财务因子只能解释其中一部分信息，因此 R² 较低在股票收益预测任务中较为常见。

其次，财务数据存在滞后性。财务指标反映的是过去经营状况，而未来季度收益往往更多受市场预期变化影响。即使某家公司基本面较好，也不代表下一期收益一定较高；反之，短期价格波动也可能由非财务因素驱动。

第三，样本期较短。本文测试集只有 4 个有效季度，回测结果容易受到个别季度行情影响，统计稳定性有限。Alpha、Beta、Sharpe 等指标在短样本下也只能作为参考，不能过度外推。

第四，市场风格会切换。价值、成长、规模、盈利等因子在不同市场阶段有效性不同。某些因子在训练期表现较好，不代表在验证期和测试期持续有效，这会造成模型排序能力波动。

第五，财务数据存在极端值和口径异常。负 PE、负 PCF、爆炸增长率等问题会降低模型稳定性。虽然本文使用缩尾、稳健标准化和 signed_log1p 进行处理，但这些方法无法完全消除财务口径差异。

第六，标签噪声较高。`future_return_1q` 是下一期收益，短期个股收益波动较大，其中大量成分可能无法由财务因子解释。当前特征维度也仍然有限，尚未充分纳入行业、动量、分析师预期、资金流、新闻文本和宏观状态切换等信息。

最后，回测尚未完全纳入真实交易约束，包括真实交易成本、冲击成本、停牌、涨跌停、ST 和流动性限制。因此，回测收益可能高估实际可执行表现。评级收益不完全单调也说明综合评分可以区分高低组，但不能稳定区分每一档。综合来看，本文更合理的定位是构建一个弱信号条件下的多指标辅助决策框架，而非高精度收益率预测模型。


# 7. 排序能力与分层收益分析

为避免只依赖回归误差评价模型，本文使用 Spearman RankIC 评价每个季度预测分数与未来收益排序的一致性。RankIC 为正说明高预测分数股票在下一期更倾向于获得较高收益，代表模型具有一定横截面排序能力。

分层收益进一步检验模型是否能将股票分成高低收益组。若最高分组收益显著高于最低分组，说明模型可以用于 Top-K 候选池筛选；但如果中间层不完全单调，则不应将模型解释为严格的等级收益体系。

![图7 逐季度RankIC](figures/fig07_quarterly_rankic.png)

**图表解读。** 图 7 展示了逐季度 RankIC。可以看到 RankIC 存在波动，说明信号并不稳定；但多个季度 RankIC 为正，表明模型仍有一定排序信息。该结果支持“弱排序信号”的结论。

![图8 分层收益](figures/fig08_layered_return.png)

**图表解读。** 图 8 展示了分层收益结果。最高组收益通常高于最低组，但中间层并不完全单调。这意味着模型更适合做 Top-K 股票筛选，而不是直接构造严格单调的完整评级解释。


# 8. PROMETHEE 综合评价与 Hybrid Score

PROMETHEE 是一种多准则决策方法，可将多个方向不同、量纲不同的指标转化为综合净流评分。本文使用熵权法计算指标客观权重，再在每个季度内部进行 PROMETHEE 评分。整个 PROMETHEE 过程不使用 `future_return_1q`，收益标签只用于后续事后检验评分有效性。

Hybrid Score 将 PROMETHEE 基本面综合评分与机器学习排序信号结合。本文设置了 `pure_promethee_score`、`pure_ml_score`、`hybrid_score_70_30`、`hybrid_score_50_50`、`hybrid_score_30_70` 和 `robust_hybrid_score` 等版本。所有权重方案均为事前固定，不使用测试集表现反向调权。PROMETHEE 主评分采用 `without_scale` 版本，`with_scale` 作为稳健性对照。

![图12 PROMETHEE规模版本相关性](figures/fig12_promethee_with_without_scale_corr.png)

**图表解读。** 图 12 比较了 PROMETHEE `with_scale` 与 `without_scale` 两个版本的净流分数。两者相关性较高，说明主要基本面指标带来的排序信息较为一致；但规模因子仍会改变部分个股排序。为避免规模风格暴露过强，本文主评分采用不含规模因子的版本，含规模版本用于稳健性对照。


# 9. 策略回测与绩效指标

## 9.1 回测设定

每个季度内部按综合评分或模型预测分数排序，分别构建 Top10、Top20 和 Top50 等权组合，并与全市场等权基准进行比较。验证集用于方案比较，测试集作为最终 holdout 检验。本文不使用测试集表现反向调节模型权重或评分权重。

## 9.2 绩效指标定义

| 指标 | 含义 | 计算说明 |
|---|---|---|
| 持有期年化收益率 | 组合在持有期内折算到年度的收益 | 若季度收益为 r_t，则 `annual_return = Π(1+r_t)^(4/n)-1` |
| 年化波动率 | 组合收益波动程度 | `quarterly_std × sqrt(4)` |
| Alpha 值 | 策略相对基准的超额收益截距 | 对策略超额收益与基准超额收益做 CAPM 回归 |
| Beta 值 | 策略对基准波动的敏感度 | CAPM 回归中的市场系数 |
| 夏普比率 | 单位风险收益 | `(annual_return - risk_free_rate) / annual_volatility` |
| 最大回撤 | 累计净值从峰值到谷值的最大跌幅 | 基于累计净值曲线计算 |
| 正收益期占比 | 收益为正的季度比例 | `positive_periods / total_periods` |
| 跑赢基准比例 | 策略收益高于基准收益的季度比例 | `outperform_periods / total_periods` |
| 换手率 | 组合持仓变化程度 | 相邻季度持仓变化比例 |

本文没有单独无风险利率数据，因此默认 `risk_free_rate = 0`。如果收益数据以百分数口径存储，则在计算前转换为小数。由于测试集只有 4 个有效季度，Alpha、Beta 和 Sharpe 的统计稳定性有限，应作为辅助指标而非绝对结论。


In [1]:
import pandas as pd
from pathlib import Path

perf_path = Path("final_performance_metrics.csv")
if not perf_path.exists():
    perf_path = Path("outputs/final_performance_metrics.csv")
perf = pd.read_csv(perf_path)
core = perf[
    (
        (perf["score_version"].eq("hybrid_score_50_50") & perf["topk"].eq(20))
        | (perf["score_version"].eq("hybrid_score_30_70") & perf["topk"].eq(10))
        | (perf["score_version"].eq("pure_ml_score") & perf["topk"].eq(20))
        | perf["score_version"].eq("benchmark")
    )
].copy()
cols = [
    "split",
    "score_version",
    "topk",
    "role",
    "holding_period_annualized_return",
    "annualized_volatility",
    "alpha_annualized",
    "beta",
    "sharpe_ratio",
    "max_drawdown",
    "positive_period_ratio",
    "outperform_benchmark_ratio",
    "turnover_rate",
]
display(core[cols].sort_values(["split", "role", "score_version", "topk"]))


,split,score_version,topk,role,holding_period_annualized_return,annualized_volatility,alpha_annualized,beta,sharpe_ratio,max_drawdown,positive_period_ratio,outperform_benchmark_ratio,turnover_rate
0,test,benchmark,NaN,全市场等权基准,0.031166,0.150187,NaN,NaN,0.207514,-0.004145,0.50,NaN,0.000000
1,test,hybrid_score_30_70,10.0,收益最优对照,0.330898,0.252020,0.287024,1.503385,1.312982,0.000000,0.75,0.75,0.633333
5,test,hybrid_score_50_50,20.0,最终主方案,0.160994,0.162297,0.127044,1.031888,0.991970,0.000000,0.75,1.00,0.500000
11,test,pure_ml_score,20.0,纯机器学习对照,0.287445,0.170658,0.256203,0.921269,1.684333,0.000000,0.75,1.00,0.333333
19,valid,benchmark,NaN,全市场等权基准,0.186334,0.205878,NaN,NaN,0.905074,-0.094446,0.75,NaN,0.000000
20,valid,hybrid_score_30_70,10.0,收益最优对照,0.584718,0.195997,0.445299,0.604337,2.983299,0.000000,1.00,0.75,0.766667
24,valid,hybrid_score_50_50,20.0,最终主方案,0.438933,0.170303,0.319255,0.545410,2.577372,0.000000,1.00,0.50,0.666667
30,valid,pure_ml_score,20.0,纯机器学习对照,0.339887,0.197359,0.171072,0.818759,1.722172,-0.008689,0.75,0.75,0.416667


## 9.3 回测结果解读

![图9 Top-K累计收益](figures/fig09_composite_topk_cumulative_return.png)

**图表解读。** 图 9 对比了最终主方案、收益最优对照和纯机器学习对照的累计收益曲线。`hybrid_score_30_70 Top10` 在收益上更激进，但持仓更集中。`hybrid_score_50_50 Top20` 在收益、分散度和解释性之间更均衡，因此被选为最终主方案。

![图10 综合评分策略收益对比](figures/fig10_composite_strategy_return_compare.png)

**图表解读。** 图 10 展示了不同综合评分版本和 Top-K 组合在测试集上的累计收益对比。可以看到，不同评分权重和 Top-K 选择对收益影响明显。本文没有因为某个组合测试集收益最高就直接选择它，而是结合验证集表现、分散度和解释性综合判断。

![图11 收益与换手率权衡](figures/fig11_return_turnover_tradeoff.png)

**图表解读。** 图 11 展示收益与换手率之间的权衡。Hybrid 方法改善了解释性，并在部分方案上改善波动表现，但并未显著降低换手率。真实交易中还需要进一步加入换手约束、交易成本、流动性和可交易性过滤。


# 10. 评级体系检验

本文基于最终综合评分构建 13 级评级体系，评级从高到低为 `AAA, AA+, AA, AA-, A+, A, A-, BBB+, BBB, BBB-, BB+, BB, B`。评级在每个季度内部按综合评分分位数划分，`future_return_1q` 只用于事后检验，不参与评级划分。

评级结果显示，各等级样本分布整体较均衡，高评级组多数情况下优于低评级组。但评级收益并不完全单调，说明综合评分对高低组有区分能力，却不能稳定解释每一档评级的收益差异。因此，评级更适合作为候选池筛选工具，而不是严格单调的收益等级体系。

![图13 评级有效性](figures/fig13_rating_effectiveness.png)

**图表解读。** 图 13 展示不同评级在测试期的平均未来收益。高评级组整体具备一定选股价值，但评级之间并非严格递减或递增。该现象与分层收益分析一致，说明模型和综合评分更适合 Top-K 筛选，而非完整单调评级解释。


# 11. 最终策略选择

**最终主方案：** `hybrid_score_50_50 + Top20`  
**收益最优对照：** `hybrid_score_30_70 + Top10`  
**纯机器学习对照：** `pure_ml_score / Random Forest Top20`

最终主方案没有直接选择收益最高的 `hybrid_score_30_70 Top10`，主要有五点原因。第一，Top10 持仓更集中，个股风险更高，组合更容易受到少数股票影响。第二，测试期只有 4 个有效季度，收益最优结果可能受到样本偶然性影响。第三，课程大作业更强调完整建模框架和可解释性，而不是只追求短期回测收益最高。第四，`hybrid_score_50_50` 同时结合 PROMETHEE 基本面综合评价与机器学习排序信号，解释性强于纯机器学习模型。第五，Top20 的分散度更好，更适合作为报告中的稳健主方案。

如果从纯收益角度看，`hybrid_score_30_70 Top10` 可以作为激进方案；如果从解释性、稳健性和课程报告完整性角度看，`hybrid_score_50_50 Top20` 更适合作为最终推荐方案。需要强调的是，Hybrid Score 并未显著降低换手率，因此真实交易中仍需加入换手约束和交易成本约束。


# 12. 结论与不足

本文完成了完整的数据审计、标签验证、防泄漏、建模、综合评价和回测流程。数据处理层面，本文解决了标签前移验证、收益字段泄漏、宏观因子数量冲突、时间切分和 train-only 预处理问题。模型层面，回归预测能力较弱，但 RankIC 和分层收益显示存在弱排序信号。PROMETHEE 和 Hybrid Score 提升了模型体系的解释性和多指标综合评价能力。

最终，本文推荐 `hybrid_score_50_50 + Top20` 作为主方案。该方案更适合作为 Top-K 选股和辅助决策工具，而不是精确收益率预测模型。评级体系可用于候选池筛选，但不适合作为严格单调的收益等级解释。

本文仍存在若干不足。第一，测试期只有 4 个有效季度，统计稳定性有限。第二，未充分考虑真实交易成本、冲击成本、停牌、涨跌停、ST 和流动性约束。第三，评级收益不完全单调，说明信号强度不足以稳定区分所有评级档位。第四，Alpha、Beta、Sharpe 等指标在短样本下稳定性有限。第五，财务因子对短期收益解释力有限，后续可以加入行业中性化、动量因子、宏观状态、资金流、分析师预期、交易约束和滚动训练机制。
